# Занятие 3. Введение в Pandas



---
## 1. Повторение: словари и множества

### Словарь (dict)

Словарь — неупорядоченная коллекция пар **ключ → значение**. Ключи уникальны и хэшируемы (строки, числа, кортежи). Значения могут быть любыми, в том числе другими словарями.

Основные операции:
- создание: `{ключ: значение, ...}` или `dict()`
- доступ: `d[key]` — выбрасывает `KeyError` если нет; `d.get(key, default)` — безопасно
- добавление/изменение: `d[key] = value`
- итерация: `d.items()` (пары), `d.keys()`, `d.values()`
- проверка: `key in d`

**Вложенные словари** — ключ ведёт к другому словарю. Удобно для хранения нескольких атрибутов одной сущности.

> **Зачем это в контексте pandas?**  
> Словари — таблицы соответствий для создания новых столбцов, источник данных при создании DataFrame, аргументы методов (`rename`, `agg`). Чем лучше вы знаете словари, тем проще работать с pandas.

### Множество (set)

Множество — неупорядоченная коллекция **уникальных** элементов. Главные применения: устранение дубликатов, быстрая проверка принадлежности (O(1)), теоретико-множественные операции (`&`, `|`, `-`).


In [1]:
# Словари: основные операции
royalties = {
    'Rock': 0.004,
    'Pop':  0.003,
    'Jazz': 0.005,
    'Classical': 0.006,
    'Hip-Hop': 0.0035,
    'Electronic': 0.0038,
}

# Доступ к значению
print('Royalty for Rock:', royalties['Rock'])
print('Royalty for Folk (с дефолтом):', royalties.get('Folk', 0.003))

Royalty for Rock: 0.004
Royalty for Folk (с дефолтом): 0.003


In [2]:
# Вложенный словарь
platform_info = {
    'Spotify':       {'country': 'Sweden',  'paid_tier': True,  'monthly_users_m': 602},
    'Apple Music':   {'country': 'USA',     'paid_tier': True,  'monthly_users_m': 88},
    'YouTube Music': {'country': 'USA',     'paid_tier': False, 'monthly_users_m': 80},
}

print('\nSpotify users (M):', platform_info['Spotify']['monthly_users_m'])


Spotify users (M): 602


In [ ]:
# Итерация
for platform, info in platform_info.items():
    print(f"  {platform}: платный = {info['paid_tier']}")

  Spotify: платный = True
  Apple Music: платный = True
  YouTube Music: платный = False


In [ ]:
# Множества: операции и быстрая проверка
streaming_platforms = {'Spotify', 'Apple Music', 'YouTube Music', 'Deezer', 'Tidal'}
paid_platforms      = {'Spotify', 'Apple Music', 'Tidal'}

print('Только бесплатные:', streaming_platforms - paid_platforms)
print('Все платформы:',     streaming_platforms | paid_platforms)
print('Пересечение:',       streaming_platforms & paid_platforms)

Только бесплатные: {'Deezer', 'YouTube Music'}
Все платформы: {'Deezer', 'YouTube Music', 'Spotify', 'Tidal', 'Apple Music'}
Пересечение: {'Spotify', 'Tidal', 'Apple Music'}


In [ ]:
# Быстрая проверка — особенно полезно в циклах
print('\n"SoundCloud" в наборе?', 'SoundCloud' in streaming_platforms)
print('"Spotify" в наборе?',       'Spotify'    in streaming_platforms)


"SoundCloud" в наборе? False
"Spotify" в наборе? True


---
## 2. Что такое pandas и зачем он нужен

**pandas** — библиотека Python для работы с табличными и временными данными. Её название происходит от *panel data* (панельные данные — экономический термин для многомерных датасетов).

Pandas позволяет:
- загружать данные из CSV, Excel, SQL, JSON и других форматов;
- исследовать структуру датасета;
- фильтровать, сортировать, группировать, агрегировать;
- работать с пропущенными значениями;
- работать с временными рядами.

Стандартный импорт:


In [ ]:
import pandas as pd
import numpy as np

print('pandas:', pd.__version__)
print('numpy: ', np.__version__)


pandas: 2.2.3
numpy:  2.2.1


---
## 3. Series и DataFrame

### Series

**Series** — одномерный массив с индексом. Можно воспринимать как один столбец таблицы. Поддерживает поэлементные арифметические операции и методы агрегации.

### DataFrame

**DataFrame** — двумерная таблица: строки (записи) и столбцы (переменные). Каждый столбец — это Series с общим индексом. DataFrame можно создать из словаря, списка списков, другого DataFrame или загрузив файл.

**Индекс** — метки строк (по умолчанию 0, 1, 2, …). Можно задать любой столбец как индекс.


In [ ]:
# Series
s = pd.Series([120000, 85000, 43000, 9000], index=['Spotify', 'Apple Music', 'Deezer', 'Tidal'])
print('=== Series: прослушивания по платформам ===')
print(s)
print('\nСумма:  ', s.sum())
print('Среднее:', s.mean())
print('Макс:   ', s.max(), '-', s.idxmax())


=== Series: прослушивания по платформам ===
Spotify        120000
Apple Music     85000
Deezer          43000
Tidal            9000
dtype: int64

Сумма:   257000
Среднее: 64250.0
Макс:    120000 - Spotify


In [ ]:
# DataFrame из словаря
mini_df = pd.DataFrame({
    'track_id':  ['T100', 'T101', 'T102'],
    'artist':    ['Luna Park', 'Stone Echo', 'Mira Sol'],
    'streams':   [59013, 25061, 88000],
    'genre':     ['Rock', 'Folk', 'Pop'],
})
print('=== Мини-датафрейм ===')
print(mini_df)
print('\nТип объекта:', type(mini_df))
print('Тип столбца:', type(mini_df['streams']))


=== Мини-датафрейм ===
  track_id      artist  streams genre
0     T100   Luna Park    59013  Rock
1     T101  Stone Echo    25061  Folk
2     T102    Mira Sol    88000   Pop

Тип объекта: <class 'pandas.core.frame.DataFrame'>
Тип столбца: <class 'pandas.core.series.Series'>


---
## 4. Чтение данных: `read_csv`

`pd.read_csv()` — основной способ загрузки табличных данных. Принимает путь к файлу или URL.

Наиболее полезные параметры:
- `sep` — разделитель (по умолчанию `,`)
- `encoding` — кодировка файла
- `parse_dates` — список столбцов, которые нужно сразу разобрать как даты
- `index_col` — столбец, который станет индексом
- `nrows` — читать только первые N строк (полезно при разведке большого файла)


#### Другие способы загрузки данных

pandas умеет читать данные из множества форматов — не только из CSV.

**Excel** (`pd.read_excel`): читает файлы `.xlsx` и `.xls`. Можно указать конкретный лист через параметр `sheet_name` (по умолчанию читается первый). Требует установленной библиотеки `openpyxl`.

**JSON** (`pd.read_json`): читает JSON-файлы и строки. Хорошо работает, когда JSON представляет собой список объектов — каждый объект становится строкой датафрейма.

**SQL** (`pd.read_sql`): выполняет SQL-запрос к базе данных и возвращает результат как датафрейм. Требует объект подключения (например, из библиотек `sqlite3` или `sqlalchemy`).

**Буфер обмена** (`pd.read_clipboard`): читает данные, скопированные в буфер — удобно, если нужно быстро перенести небольшую таблицу из браузера или Excel.

**Из словаря или списка** (`pd.DataFrame(...)`): датафрейм можно создать прямо в коде — из словаря (ключи → названия столбцов) или из списка списков с отдельным указанием `columns`.

Все перечисленные функции поддерживают большинство тех же параметров, что и `read_csv`: `index_col`, `usecols` (выбрать только нужные столбцы), `nrows`, `dtype` и другие.

## Датасет для примеров

На протяжении всего занятия мы будем работать с одним датасетом — **статистикой прослушиваний музыкальных треков** на стриминговых платформах за 2023 год.

Файл `music_streams.csv` содержит 200 записей со следующими столбцами:

| Столбец | Описание |
|---|---|
| `track_id` | Уникальный ID трека |
| `artist` | Исполнитель |
| `genre` | Жанр |
| `platform` | Платформа (Spotify, Apple Music и др.) |
| `country` | Страна прослушивания |
| `streams` | Количество прослушиваний |
| `duration_sec` | Длительность трека в секундах |
| `date` | Дата события |
| `is_explicit` | Наличие ненормативного контента |
| `release_year` | Год выхода трека |


In [ ]:
# Загружаем наш датасет
music = pd.read_csv('music_streams.csv')

print('Тип объекта:', type(music))
print('Форма (строки, столбцы):', music.shape)


Тип объекта: <class 'pandas.core.frame.DataFrame'>
Форма (строки, столбцы): (200, 10)


---
## 5. Первичное знакомство с данными

После загрузки данных первый шаг — понять их структуру: сколько строк и столбцов, какие типы данных, есть ли пропуски, как распределены значения.

Ключевые методы:
- `head(n)` / `tail(n)` — первые / последние n строк
- `shape` — кортеж (строки, столбцы)
- `columns` — список названий столбцов
- `dtypes` — типы данных каждого столбца
- `info()` — сводка: типы + количество непустых значений
- `describe()` — описательные статистики (count, mean, std, min, quartiles, max) для числовых столбцов
- `value_counts()` — частоты значений в столбце
- `nunique()` — количество уникальных значений
- `unique()` — массив уникальных значений


In [ ]:
# Первый взгляд на данные
display(music.head()) # по умолчанию - 5 строк

,track_id,artist,genre,platform,country,streams,duration_sec,date,is_explicit,release_year
0,T1000,Luna Park,Rock,YouTube Music,France,59013,191,2023-11-24,True,2023
1,T1001,Luna Park,Folk,Spotify,Russia,25061,231,2023-10-07,True,2022
2,T1002,The Waves,Classical,Tidal,UK,58287,349,2023-11-05,False,2018
3,T1003,Max Orbit,Electronic,YouTube Music,Germany,56943,292,2023-03-23,True,2018
4,T1004,Luna Park,Electronic,YouTube Music,Brazil,11890,355,2023-07-14,True,2021


In [ ]:
print('=== Форма датафрейма ===')
print(music.shape)

print('\n=== Столбцы ===')
print(music.columns.tolist())

print('\n=== Типы данных ===')
print(music.dtypes)


=== Форма датафрейма ===
(200, 10)

=== Столбцы ===
['track_id', 'artist', 'genre', 'platform', 'country', 'streams', 'duration_sec', 'date', 'is_explicit', 'release_year']

=== Типы данных ===
track_id        object
artist          object
genre           object
platform        object
country         object
streams          int64
duration_sec     int64
date            object
is_explicit       bool
release_year     int64
dtype: object


In [ ]:
print('=== Описательные статистики ===')
music.describe()

=== Описательные статистики ===


,streams,duration_sec,release_year
count,200.000000,200.000000,200.000000
mean,77158.900000,277.045000,2020.665000
std,41110.467844,90.011518,1.722676
min,1172.000000,124.000000,2018.000000
25%,46656.500000,193.750000,2019.000000
50%,78462.500000,288.500000,2021.000000
75%,108295.250000,358.250000,2022.000000
max,148279.000000,416.000000,2023.000000


In [ ]:
print('=== Уникальные жанры ===')
print(music['genre'].unique())

print('\n=== Количество записей по платформам ===')
print(music['platform'].value_counts())

=== Уникальные жанры ===
['Rock' 'Folk' 'Classical' 'Electronic' 'Pop' 'Hip-Hop' 'Jazz' 'R&B']

=== Количество записей по платформам ===
platform
YouTube Music    46
Spotify          45
Tidal            41
Deezer           35
Apple Music      33
Name: count, dtype: int64


### 🔹 Мини-задача 5.1

Используя загруженный датафрейм `music`:
1. Выведите последние 3 строки.
2. Узнайте, сколько уникальных исполнителей (`artist`) в датасете.
3. Выведите частоты по столбцу `country` — в каких странах больше всего прослушиваний?


In [ ]:
# YOUR CODE HERE


---
## 6. Работа со столбцами

### Доступ к столбцу

Один столбец возвращает **Series**: `df['column']`  
Несколько столбцов возвращает **DataFrame**: `df[['col1', 'col2']]`

### Создание нового столбца

Новый столбец создаётся присваиванием. Правая часть может быть:
- арифметическим выражением над другими столбцами (поэлементно),
- константой,
- заранее подготовленным списком или Series.

### Переименование

`df.rename(columns={'old': 'new'}, inplace=True)` — `inplace=True` изменяет объект на месте, без создания копии.

### Удаление столбца

`df.drop(columns=['col'], inplace=True)`


In [ ]:
# Создание нового столбца: длительность в минутах
music['duration_min'] = music['duration_sec'] / 60

# Создание нового столбца: потоки в тысячах (для удобства отображения)
music['streams_k'] = music['streams'] / 1000

print('Новые столбцы добавлены:')
display(music[['track_id', 'artist', 'streams', 'streams_k', 'duration_sec', 'duration_min']].head())


Новые столбцы добавлены:


,track_id,artist,streams,streams_k,duration_sec,duration_min
0,T1000,Luna Park,59013,59.013,191,3.183333
1,T1001,Luna Park,25061,25.061,231,3.850000
2,T1002,The Waves,58287,58.287,349,5.816667
3,T1003,Max Orbit,56943,56.943,292,4.866667
4,T1004,Luna Park,11890,11.890,355,5.916667


In [ ]:
# Переименование столбца
music.rename(columns={'is_explicit': 'explicit'}, inplace=True)
print('Столбцы после переименования:')
print(music.columns.tolist())


Столбцы после переименования:
['track_id', 'artist', 'genre', 'platform', 'country', 'streams', 'duration_sec', 'date', 'explicit', 'release_year', 'duration_min', 'streams_k']


---
## 7. Преобразование типов: `astype` и `pd.to_datetime`

При загрузке CSV pandas старается определить тип каждого столбца автоматически, но иногда нужно явное преобразование.

`astype(тип)` — преобразует Series к указанному типу. Стандартные типы Python (`int`, `float`, `str`) работают, а также строки типа `'float64'`, `'int32'`, `'bool'` и т. д.

### Даты: `pd.to_datetime`

Даты часто хранятся как строки (`'2023-12-02'`). Чтобы работать с ними полноценно — сортировать, вычислять разности, ресемплировать — нужно преобразовать столбец к типу `datetime64`.

После преобразования открывается **аксессор `.dt`**, через который доступны компоненты даты: год, месяц, день, день недели, час, квартал и другие.


In [ ]:
# Проверим текущий тип столбца date
print('Тип до преобразования:', music['date'].dtype)
print('Пример значения:      ', music['date'].iloc[0], type(music['date'].iloc[0]))

# Преобразуем к datetime
music['date'] = pd.to_datetime(music['date'])

print('\nТип после преобразования:', music['date'].dtype)
print('Пример значения:          ', music['date'].iloc[0], type(music['date'].iloc[0]))


Тип до преобразования: object
Пример значения:       2023-11-24 <class 'str'>

Тип после преобразования: datetime64[ns]
Пример значения:           2023-11-24 00:00:00 <class 'pandas._libs.tslibs.timestamps.Timestamp'>


In [ ]:
# Аксессор .dt: компоненты даты
music['month']      = music['date'].dt.month
music['day_of_week'] = music['date'].dt.dayofweek   # 0=пн, 6=вс
music['quarter']    = music['date'].dt.quarter

print('Добавленные столбцы:')
display(music[['date', 'month', 'day_of_week', 'quarter']].head())


Добавленные столбцы:


,date,month,day_of_week,quarter
0,2023-11-24,11,4,4
1,2023-10-07,10,5,4
2,2023-11-05,11,6,4
3,2023-03-23,3,3,1
4,2023-07-14,7,4,3


### 🔹 Мини-задача 7.1

Используя датафрейм `music`:
1. Выведите минимальную и максимальную дату в датасете.
2. Добавьте столбец `week` — номер недели года (подсказка: `.dt.isocalendar().week`).
3. Сколько записей приходится на каждый квартал? Используйте `value_counts()`.


In [ ]:
# YOUR CODE HERE


---
## 8. Итерация и маппинг: цикл по столбцу, `.map()`, `.apply()`

Часто нужно для каждой строки найти значение в словаре или применить произвольную функцию.

### Способ 1: цикл + список

Классический Python-подход: итерируемся по столбцу (Series — итерируемый объект), собираем список, присваиваем. Прозрачно, но медленно на больших данных.

### Способ 2: `.map(словарь)`

Когда нужно просто «перевести» значения одного столбца через словарь — `.map()` лаконичен и быстрее цикла. Если ключ не найден, ставит `NaN`.

### Способ 3: `.apply(функция)`

Применяет произвольную функцию к каждому элементу Series. Подходит для любой логики, которую нельзя выразить прямой арифметикой. Принимает обычную функцию или lambda.

**Итог**: цикл — понятно; `.map()` — быстро для словарей; `.apply()` — гибко для логики.


In [ ]:
# Способ 1: цикл — добавить роялти-ставку жанра из словаря
royalties = {
    'Rock': 0.004, 'Pop': 0.003, 'Jazz': 0.005, 'Classical': 0.006,
    'Hip-Hop': 0.0035, 'Electronic': 0.0038, 'Folk': 0.0032, 'R&B': 0.0033,
}

royalty_list = []
for genre in music['genre']:
    royalty_list.append(royalties.get(genre, 0.003))

music['royalty_rate'] = royalty_list

print('Первые строки с роялти:')
display(music[['track_id', 'genre', 'streams', 'royalty_rate']].head())


Первые строки с роялти:


,track_id,genre,streams,royalty_rate
0,T1000,Rock,59013,0.0040
1,T1001,Folk,25061,0.0032
2,T1002,Classical,58287,0.0060
3,T1003,Electronic,56943,0.0038
4,T1004,Electronic,11890,0.0038


In [ ]:
# Способ 2: .map() — тот же результат, короче
music['royalty_rate_map'] = music['genre'].map(royalties)

# Проверяем, что результаты одинаковые
print('Результаты совпадают:', (music['royalty_rate'] == music['royalty_rate_map']).all())

# Вычислим доход от трека
music['revenue'] = music['streams'] * music['royalty_rate']
display(music[['track_id', 'artist', 'streams', 'royalty_rate', 'royalty_rate_map', 'revenue']].head())


Результаты совпадают: True


,track_id,artist,streams,royalty_rate,royalty_rate_map,revenue
0,T1000,Luna Park,59013,0.0040,0.0040,236.0520
1,T1001,Luna Park,25061,0.0032,0.0032,80.1952
2,T1002,The Waves,58287,0.0060,0.0060,349.7220
3,T1003,Max Orbit,56943,0.0038,0.0038,216.3834
4,T1004,Luna Park,11890,0.0038,0.0038,45.1820


In [ ]:
# Способ 3: .apply() — категоризация по количеству стримов
def stream_tier(n):
    if n < 5000:
        return 'niche'
    elif n < 50000:
        return 'mid'
    else:
        return 'viral'

music['stream_tier'] = music['streams'].apply(stream_tier)

print('Распределение по уровням популярности:')
print(music['stream_tier'].value_counts())


Распределение по уровням популярности:
stream_tier
viral    148
mid       47
niche      5
Name: count, dtype: int64


### 🔹 Мини-задача 8.1

Дан словарь `genre_origin` — континент происхождения жанра:

```
genre_origin = {
    'Rock': 'Europe', 'Pop': 'Global', 'Jazz': 'Americas',
    'Classical': 'Europe', 'Hip-Hop': 'Americas',
    'Electronic': 'Europe', 'Folk': 'Global', 'R&B': 'Americas',
}
```

1. Создайте столбец `genre_origin` с помощью **цикла**.
2. Создайте столбец `duration_category` через `.apply()`: `'short'` (< 180 сек), `'medium'` (180–300), `'long'` (> 300).
3. Выведите, сколько треков каждой `duration_category` встречается в датасете.


In [ ]:
genre_origin_map = {
    'Rock': 'Europe', 'Pop': 'Global', 'Jazz': 'Americas',
    'Classical': 'Europe', 'Hip-Hop': 'Americas',
    'Electronic': 'Europe', 'Folk': 'Global', 'R&B': 'Americas',
}

# YOUR CODE HERE


---
## 9. Фильтрация: булева индексация

Основной способ фильтрации в pandas — создать **булеву маску** (Series из `True`/`False`) и передать её в квадратные скобки.

Правила:
- Одно условие: `df[df['col'] > значение]`
- Несколько условий — **обязательно скобки** вокруг каждого:
  - И: `(условие1) & (условие2)`
  - ИЛИ: `(условие1) | (условие2)`
  - НЕ: `~(условие)`
- Принадлежность множеству: `df['col'].isin(['a', 'b'])`

### `.loc` и `.iloc`

- `.loc[строки, столбцы]` — доступ по **меткам** индекса и названиям столбцов; можно передавать булеву маску
- `.iloc[строки, столбцы]` — доступ по **позициям** (целые числа)


In [ ]:
# Простая фильтрация: треки с более чем 100 000 стримов
popular = music[music['streams'] > 100_000]
print(f'Треков с > 100k стримов: {len(popular)}')
display(popular[['track_id', 'artist', 'genre', 'streams']].head())


Треков с > 100k стримов: 64


,track_id,artist,genre,streams
6,T1006,Mira Sol,Pop,119358
7,T1007,Mira Sol,Hip-Hop,140521
10,T1010,DJ Kontrol,Hip-Hop,147659
14,T1014,The Drifters,Rock,133585
15,T1015,Mira Sol,Jazz,141894


In [ ]:
# Составная фильтрация: Rock или Electronic, стримы > 50k
mask = (music['genre'].isin(['Rock', 'Electronic'])) & (music['streams'] > 50_000)
filtered = music[mask]
print(f'Строк после фильтрации: {filtered.shape[0]}')
display(filtered[['artist', 'genre', 'streams', 'platform']].head())


Строк после фильтрации: 33


,artist,genre,streams,platform
0,Luna Park,Rock,59013,YouTube Music
3,Max Orbit,Electronic,56943,YouTube Music
13,Stone Echo,Rock,89674,Spotify
14,The Drifters,Rock,133585,YouTube Music
16,The Drifters,Rock,81112,Spotify


In [ ]:
# .loc: фильтрация + выбор конкретных столбцов
result = music.loc[music['explicit'] == True, ['artist', 'genre', 'streams']]
print(f'Explicit-треков: {len(result)}')
display(result.head())


Explicit-треков: 100


,artist,genre,streams
0,Luna Park,Rock,59013
1,Luna Park,Folk,25061
3,Max Orbit,Electronic,56943
4,Luna Park,Electronic,11890
5,Stone Echo,Electronic,18733


### 🔹 Мини-задача 9.1

1. Отфильтруйте треки, у которых `stream_tier == 'viral'`.
2. Из результата оставьте только треки с платформы `'Spotify'` или `'Apple Music'`.
3. Выведите форму итогового датафрейма и топ-3 жанра в нём (`value_counts()`).


In [ ]:
# YOUR CODE HERE


---
## 10. Группировка и агрегация: `groupby` и `agg`

**Группировка** — разбивка данных на группы по значениям одного или нескольких столбцов, с последующим вычислением агрегата для каждой группы.

Схема работы: `df.groupby('столбец')['столбец_для_агрегации'].функция()`

Часто используемые функции агрегации: `sum`, `mean`, `median`, `count`, `min`, `max`, `std`.

Метод `.agg()` позволяет применить несколько агрегаций сразу — либо списком, либо через именованные аргументы:  
`df.groupby('col').agg(имя=('столбец', 'функция'), ...)`

При группировке по нескольким столбцам результат имеет **MultiIndex** — иерархический индекс из нескольких уровней.

### Процент от общего

После агрегации легко посчитать долю каждой группы: разделить столбец на его сумму и умножить на 100.

### Сброс индекса

После `groupby` столбец-группировщик становится индексом. Чтобы вернуть его в обычные столбцы — `.reset_index()`.


In [ ]:
# Суммарное количество стримов по жанрам
streams_by_genre = music.groupby('genre')['streams'].sum().sort_values(ascending=False)
print('=== Стримы по жанрам ===')
print(streams_by_genre)


=== Стримы по жанрам ===
genre
Pop           2476897
Jazz          2299381
Folk          2096991
Electronic    1885205
R&B           1876545
Hip-Hop       1803324
Classical     1683928
Rock          1309509
Name: streams, dtype: int64


In [ ]:
# Несколько агрегатов сразу
genre_stats = music.groupby('genre').agg(
    total_streams = ('streams', 'sum'),
    avg_streams   = ('streams', 'mean'),
    median_streams = ('streams', 'median'),
    track_count   = ('streams', 'count'),
).round(0)

# Процент от общего
genre_stats['pct_of_total'] = (genre_stats['total_streams'] / genre_stats['total_streams'].sum() * 100).round(2)

print('=== Статистика по жанрам ===')
display(genre_stats.sort_values('total_streams', ascending=False))


=== Статистика по жанрам ===


,total_streams,avg_streams,median_streams,track_count,pct_of_total
genre,,,,,
Pop,2476897,88461.0,83835.0,28,16.05
Jazz,2299381,85162.0,95357.0,27,14.90
Folk,2096991,80654.0,74306.0,26,13.59
Electronic,1885205,60813.0,64514.0,31,12.22
R&B,1876545,85298.0,90200.0,22,12.16
Hip-Hop,1803324,78405.0,72010.0,23,11.69
Classical,1683928,64766.0,61134.0,26,10.91
Rock,1309509,77030.0,79515.0,17,8.49


In [ ]:
# Группировка по двум столбцам -> MultiIndex
platform_genre = music.groupby(['platform', 'genre'])['streams'].sum()
print('=== Тип индекса ===', type(platform_genre.index))
print()
print(platform_genre.head(10))


=== Тип индекса === <class 'pandas.core.indexes.multi.MultiIndex'>

platform     genre     
Apple Music  Classical     241696
             Electronic    278569
             Folk           10204
             Hip-Hop       275832
             Jazz          403403
             Pop           444136
             R&B           343029
             Rock          156442
Deezer       Classical     236038
             Electronic    370587
Name: streams, dtype: int64


### 🔹 Мини-задача 10.1

1. Сгруппируйте `music` по `platform` и вычислите: суммарные стримы, средние стримы, количество треков.
2. Добавьте столбец с долей каждой платформы от суммарных стримов (в процентах, округлить до 1 знака).
3. Отсортируйте результат по суммарным стримам по убыванию.


In [ ]:
# YOUR CODE HERE


---
## 11. Работа со строками: аксессор `.str`

Если столбец содержит строки, аксессор `.str` открывает все методы Python-строк, применяя их ко всей Series сразу (поэлементно).

Наиболее используемые методы:
- `.str.lower()` / `.str.upper()` — регистр
- `.str.contains('подстрока')` — булева маска
- `.str.startswith('...')` / `.str.endswith('...')`
- `.str.replace('что', 'на_что')` — замена
- `.str.split('разделитель')` — разбить по разделителю, возвращает столбец списков
- `.str.split('разделитель').str[N]` — взять N-й элемент после разбиения
- `.str.len()` — длина строки
- `.str.strip()` — убрать пробелы по краям

Аксессор `.str` особенно полезен при обработке столбцов вида `'HH:MM'`, `'2023-Q1'`, кодов с разделителями и т. п.


In [ ]:
# Пример: работаем с track_id вида 'T1042'
# Извлечём числовой номер трека (убрав 'T')
music['track_num'] = music['track_id'].str.replace('T', '').astype(int)

# Проверка: треки, у которых artist содержит пробел (двусложное имя)
music['is_band'] = music['artist'].str.contains(' ')

print('Пример track_num:')
display(music[['track_id', 'track_num', 'artist', 'is_band']].head(8))


Пример track_num:


,track_id,track_num,artist,is_band
0,T1000,1000,Luna Park,True
1,T1001,1001,Luna Park,True
2,T1002,1002,The Waves,True
3,T1003,1003,Max Orbit,True
4,T1004,1004,Luna Park,True
5,T1005,1005,Stone Echo,True
6,T1006,1006,Mira Sol,True
7,T1007,1007,Mira Sol,True


In [ ]:
# Создадим столбец с временем вида 'HH:MM' для демонстрации .str.split
import random
random.seed(1)
music['listen_time'] = [f'{random.randint(0,23):02d}:{random.randint(0,59):02d}' for _ in range(len(music))]

# Извлечём час как целое число
music['listen_hour'] = music['listen_time'].str.split(':').str[0].astype(int)

print('Пример столбца listen_time → listen_hour:')
display(music[['listen_time', 'listen_hour']].head(8))


Пример столбца listen_time → listen_hour:


,listen_time,listen_hour
0,04:36,4
1,02:16,2
2,03:31,3
3,14:30,14
4,20:24,20
5,06:06,6
6,15:01,15
7,12:27,12


### 🔹 Мини-задача 11.1

1. Создайте столбец `time_of_day` по значению `listen_hour`:
   - `'night'` (0–5), `'morning'` (6–11), `'day'` (12–17), `'evening'` (18–23)  
   Используйте `.apply()`.
2. Выведите, сколько прослушиваний приходится на каждое время суток.
3. В какое время суток в среднем больше всего стримов?


In [ ]:
# YOUR CODE HERE


---
## 12. Ресемплинг по времени: `resample`

`resample` — мощный инструмент для агрегации временных рядов по периодам (день, неделя, месяц и т. д.).

**Условие использования**: индекс датафрейма должен иметь тип `datetime64`. Если дата хранится в обычном столбце, нужно сделать его индексом через `.set_index('date')`.

Часто используемые коды периодов:
- `'D'` — день
- `'W'` — неделя (конец недели)
- `'ME'` — конец месяца
- `'QE'` — конец квартала

После `resample` применяется агрегирующая функция: `.sum()`, `.mean()`, `.count()` и т. д.

Если нужна сводная таблица с датами в индексе и несколькими столбцами (например, жанрами) — сначала делаем `pivot_table` или `groupby` + `unstack`, затем `resample`.


In [1]:
# Суммарные стримы по неделям
music_ts = music.set_index('date')
weekly_streams = music_ts['streams'].resample('W').sum()

print('=== Стримы по неделям (первые 8 недель) ===')
print(weekly_streams.head(8))
print(f'\nВсего недель: {len(weekly_streams)}')


NameError: name 'music' is not defined

In [ ]:
# Средние стримы по месяцам
monthly_avg = music_ts['streams'].resample('ME').mean().round(0)
print('=== Среднее количество стримов по месяцам ===')
print(monthly_avg)


=== Среднее количество стримов по месяцам ===
date
2023-01-31    102270.0
2023-02-28     84205.0
2023-03-31     70272.0
2023-04-30     82835.0
2023-05-31     82662.0
2023-06-30     68156.0
2023-07-31     61673.0
2023-08-31     85954.0
2023-09-30     68269.0
2023-10-31     75400.0
2023-11-30     65501.0
2023-12-31     86517.0
Freq: ME, Name: streams, dtype: float64


---
## 13. Квантили: `quantile`, `pd.cut`, `pd.qcut`

Часто нужно разделить числовые данные на группы — «низкие», «средние», «высокие».

### `quantile(q)` — получить пороговое значение

Возвращает значение, ниже которого лежит доля `q` всех данных. Например, `quantile(0.25)` — нижняя квартиль.

### `pd.cut(series, bins, labels)` — разбивка по **заданным** границам

Вы сами указываете пороги в `bins`. Удобно, когда пороги имеют содержательный смысл (например, «меньше 1000 стримов — нишевый»).

### `pd.qcut(series, q, labels)` — разбивка по **квантилям**

Границы определяются автоматически так, чтобы в каждой группе оказалось примерно одинаковое количество объектов. Передаётся либо число групп (`q=4`), либо список квантилей (`q=[0, 0.25, 0.75, 1.0]`).


In [ ]:
# quantile: пороги популярности
q25 = music['streams'].quantile(0.25)
q75 = music['streams'].quantile(0.75)

print(f'25-й перцентиль: {q25:,.0f} стримов')
print(f'75-й перцентиль: {q75:,.0f} стримов')


In [ ]:
# pd.cut: разбивка по вручную заданным порогам
music['popularity_cut'] = pd.cut(
    music['streams'],
    bins=[0, q25, q75, float('inf')],
    labels=['low', 'medium', 'high']
)

print('=== Разбивка pd.cut ===')
print(music['popularity_cut'].value_counts())


In [ ]:
# pd.qcut: разбивка на 3 группы (нижние 33%, средние 34%, верхние 33%)
music['popularity_qcut'] = pd.qcut(
    music['streams'],
    q=[0, 0.33, 0.67, 1.0],
    labels=['low', 'medium', 'high']
)

print('=== Разбивка pd.qcut (по квантилям) ===')
print(music['popularity_qcut'].value_counts())


### 🔹 Мини-задача 13.1

1. Найдите 33-й и 67-й перцентили столбца `duration_sec`.
2. С помощью `pd.cut` создайте столбец `dur_group` с тремя категориями: `'short'`, `'medium'`, `'long'` — по этим границам.
3. Для каждой группы вычислите среднее количество стримов (`groupby` + `mean`). Есть ли зависимость?


In [ ]:
# YOUR CODE HERE


---
## 14. Объединение датафреймов: `merge`

`pd.merge()` — аналог SQL JOIN. Объединяет два датафрейма по одному или нескольким ключевым столбцам.

Типы объединения (параметр `how`):
- `'inner'` — только строки с совпадающими ключами в **обоих** датафреймах (по умолчанию)
- `'left'` — все строки **левого** + совпадающие из правого; где нет пары — `NaN`
- `'right'` — все строки **правого** + совпадающие из левого
- `'outer'` — все строки обоих; где нет пары — `NaN`

Если ключевой столбец называется одинаково в обоих датафреймах: `on='col'`  
Если по-разному: `left_on='col_left', right_on='col_right'`


In [ ]:
# Создадим вспомогательный датафрейм с дополнительной информацией о жанрах
genre_meta = pd.DataFrame({
    'genre':       ['Rock', 'Pop', 'Jazz', 'Classical', 'Hip-Hop', 'Electronic', 'Folk', 'R&B'],
    'origin_year': [1950,   1955,  1920,   1600,        1970,       1980,         1800,   1940],
    'bpm_avg':     [130,    120,   90,     70,          95,         128,          100,    85],
    'category':    ['guitar','mainstream','jazz_classical','jazz_classical',
                    'electronic','electronic','guitar','mainstream'],
})

print('=== genre_meta ===')
display(genre_meta)


=== genre_meta ===


,genre,origin_year,bpm_avg,category
0,Rock,1950,130,guitar
1,Pop,1955,120,mainstream
2,Jazz,1920,90,jazz_classical
3,Classical,1600,70,jazz_classical
4,Hip-Hop,1970,95,electronic
5,Electronic,1980,128,electronic
6,Folk,1800,100,guitar
7,R&B,1940,85,mainstream


In [ ]:
# Left join: сохраняем все треки, добавляем метаданные жанра
music_enriched = pd.merge(music, genre_meta, on='genre', how='left')

print('Форма до merge: ', music.shape)
print('Форма после:    ', music_enriched.shape)
display(music_enriched[['track_id', 'artist', 'genre', 'streams', 'bpm_avg', 'category']].head())


Форма до merge:  (200, 25)
Форма после:     (200, 28)


,track_id,artist,genre,streams,bpm_avg,category
0,T1000,Luna Park,Rock,59013,130,guitar
1,T1001,Luna Park,Folk,25061,100,guitar
2,T1002,The Waves,Classical,58287,70,jazz_classical
3,T1003,Max Orbit,Electronic,56943,128,electronic
4,T1004,Luna Park,Electronic,11890,128,electronic


---
## 15. Сводные таблицы: `pivot_table` и `unstack`

### `pivot_table`

Создаёт сводную таблицу — аналог сводных таблиц в Excel. Основные параметры:
- `values` — что агрегировать
- `index` — что станет строками
- `columns` — что станет столбцами
- `aggfunc` — функция агрегации (`'sum'`, `'mean'`, `'count'`, и т. д.)
- `fill_value` — чем заполнить пропуски (обычно 0)

### `unstack`

Переносит уровень MultiIndex (полученного после `groupby` по нескольким столбцам) в столбцы. Результат аналогичен `pivot_table`, но строится через цепочку `groupby` → `unstack`. Удобно, когда уже есть сгруппированный результат.


In [ ]:
# pivot_table: суммарные стримы по жанрам и платформам
pivot = music.pivot_table(
    values='streams',
    index='genre',
    columns='platform',
    aggfunc='sum',
    fill_value=0
)

print('=== Стримы: жанр × платформа ===')
display(pivot)


=== Стримы: жанр × платформа ===


platform,Apple Music,Deezer,Spotify,Tidal,YouTube Music
genre,,,,,
Classical,241696,236038,352469,318171,535554
Electronic,278569,370587,401465,296656,537928
Folk,10204,321980,424774,984500,355533
Hip-Hop,275832,453800,531804,308248,233640
Jazz,403403,803126,241774,231320,619758
Pop,444136,717297,399809,534504,381151
R&B,343029,186147,584319,173470,589580
Rock,156442,2622,539419,223391,387635


In [ ]:
# unstack: тот же результат через groupby
grouped = music.groupby(['genre', 'platform'])['streams'].sum()
unstacked = grouped.unstack(level='platform', fill_value=0)

print('=== Через groupby + unstack ===')
display(unstacked.head())

# Проверим, что результаты совпадают
print('\nРезультаты идентичны:', pivot.equals(unstacked))


=== Через groupby + unstack ===


platform,Apple Music,Deezer,Spotify,Tidal,YouTube Music
genre,,,,,
Classical,241696,236038,352469,318171,535554
Electronic,278569,370587,401465,296656,537928
Folk,10204,321980,424774,984500,355533
Hip-Hop,275832,453800,531804,308248,233640
Jazz,403403,803126,241774,231320,619758



Результаты идентичны: True


---
## Шпаргалка: основные методы

| Задача | Код |
|---|---|
| Загрузить CSV | `pd.read_csv('file.csv')` |
| Первые / последние строки | `df.head()` / `df.tail()` |
| Форма и типы | `df.shape`, `df.dtypes` |
| Описательная статистика | `df.describe()` |
| Частоты значений | `df['col'].value_counts()` |
| Создать столбец (арифметика) | `df['new'] = df['a'] / df['b']` |
| Маппинг через словарь | `df['col'].map(dict)` |
| Применить функцию | `df['col'].apply(func)` |
| Строковые операции | `df['col'].str.split(':').str[0]` |
| Преобразовать дату | `pd.to_datetime(df['col'])` |
| Компоненты даты | `df['col'].dt.year`, `.dt.month`, `.dt.hour` |
| Фильтрация | `df[(df['a'] > 1) & (df['b'] == 'x')]` |
| Принадлежность | `df['col'].isin(['a', 'b'])` |
| Группировка | `df.groupby('col')['val'].agg(...)` |
| Сброс индекса | `.reset_index()` |
| Квантиль | `df['col'].quantile(0.25)` |
| Разбивка по границам | `pd.cut(df['col'], bins=[...], labels=[...])` |
| Разбивка по квантилям | `pd.qcut(df['col'], q=[0, .25, .75, 1])` |
| Сводная таблица | `df.pivot_table(values, index, columns, aggfunc)` |
| MultiIndex → столбцы | `grouped.unstack()` |
| Ресемплинг | `df.set_index('date').resample('W').sum()` |
| Объединение датафреймов | `pd.merge(left, right, on='key', how='left')` |
| Переименовать столбцы | `df.rename(columns={'old': 'new'}, inplace=True)` |
| Удалить столбец | `df.drop(columns=['col'], inplace=True)` |



**На следующем занятии:** визуализация данных — строим графики с помощью Matplotlib и Seaborn.

---
*Полезные ссылки:*
- [pandas документация](https://pandas.pydata.org/docs/)
- [pandas cheat sheet (официальный PDF)](https://pandas.pydata.org/Pandas_Cheat_Sheet.pdf)
- [pandas getting started (туториалы)](https://pandas.pydata.org/docs/getting_started/intro_tutorials/index.html)


In [ ]:
arr = np.array([1, 6, 3, 8])
np.where(arr < 5, 0, 1)

array([0, 1, 0, 1])